# Robustness pipeline -- amiri, camargo, bukhsh

Retrains each model's already-HPO'd winning config with 2 new seeds (synthetic + ssd only).

## Part 1 -- Camargo, new seeds


# Camargo robustness: retrain the HPO-selected best config with 2 new seeds

For each dataset and each seed in `NEW_SEEDS`: loads the winning architecture from the completed HPO run, pins every HPO-searched hyperparameter to that value, retrains from scratch, predicts, and saves metrics + model to `robustness/camargo/{synthetic,ssd}/<dataset>/seed_<N>/`.

Requires HPO to already be complete for that dataset.

In [ ]:
from pathlib import Path
import sys, json, shutil
import numpy as np
import pandas as pd
import pm4py

ROOT = Path.cwd().resolve().parent.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "analysis"))
sys.path.insert(0, str(ROOT / "steady_state_detection"))
sys.path.insert(0, str(ROOT / "GenerativeLSTM" / "GenerativeLSTM"))

from setttings import set_global_seed
from ts_comparison import load_splits
from camargo.trainer import CamargoTrainer
from camargo.params import default_params as camargo_default_params
import tensorflow as tf

RESULTS = ROOT / "results"
ROBUST = ROOT / "robustness" / "camargo"
GLSTM_OUT = ROOT / "GenerativeLSTM" / "GenerativeLSTM" / "output_files"

SYNTH_DATASETS = [p.stem for p in sorted((ROOT / "data" / "synthetic").glob("*.xes")) if "recency" not in p.stem]
REAL_DATASETS = ["bpic12-a", "bpic15-1", "bpic15-2", "bpic17-o",
                 "bpic20-dom", "bpic20-int", "helpdesk", "sepsis"]

NEW_SEEDS = [43, 44]

print(f"{len(SYNTH_DATASETS)} synthetic datasets, {len(REAL_DATASETS)} real-life (ssd) datasets, seeds={NEW_SEEDS}")

In [ ]:
def _load_camargo_winning_params(trim_dir: str, run_name: str) -> dict:
    """Loads the winning trial's architecture params, filling dropout/learning_rate from a fixed default when absent."""
    p = GLSTM_OUT / trim_dir / run_name / "parameters" / "model_parameters.json"
    saved = json.loads(p.read_text())

    return {
        "model_type": saved["model_type"], "lstm_act": saved["lstm_act"],
        "dense_act": saved["dense_act"], "n_size": saved["n_size"],
        "l_size": saved["l_size"], "norm_method": saved["norm_method"],
        "optim": saved["optim"], "dropout": saved.get("dropout", _CAMARGO_DROPOUT),
        "learning_rate": saved.get("learning_rate", _CAMARGO_LEARNING_RATE),
    }


_CAMARGO_DROPOUT = 0.2
_CAMARGO_LEARNING_RATE = 0.002


def _build_fixed_camargo_params(run_name: str, winning: dict) -> dict:
    """default_params() with every HPO-searched key pinned to a single value, max_eval=1."""
    p = camargo_default_params(run_name, max_eval=1, epochs=200)
    p["model_type"] = [winning["model_type"]]
    p["lstm_act"] = [winning["lstm_act"]]
    p["dense_act"] = [winning["dense_act"]]
    p["n_size"] = [winning["n_size"]]
    p["l_size"] = [winning["l_size"]]
    p["norm_method"] = [winning["norm_method"]]
    p["optim"] = [winning["optim"]]
    p["dropout"] = [winning["dropout"]]
    p["learning_rate"] = [winning["learning_rate"]]
    return p

In [ ]:
def run_camargo_robustness_one(dataset: str, is_real: bool, seed: int):
    """One (dataset, seed) robustness run. Skips if metrics already exist."""
    sub = "ssd" if is_real else "synthetic"
    out_dir = ROBUST / sub / dataset / f"seed_{seed}"
    metrics_path = out_dir / "metrics.csv"
    if metrics_path.exists():
        print(f"  [skip] {dataset}/seed_{seed}: metrics already exist")
        return

    winning_run_name = f"{dataset}_ssd_hpo" if is_real else f"{dataset}_hpo"
    winning_trim_dir = "ssd" if is_real else "hpo"
    winning_params_path = GLSTM_OUT / winning_trim_dir / winning_run_name / "parameters" / "model_parameters.json"
    if not winning_params_path.exists():
        print(f"  [skip] {dataset}: no trained camargo model at {winning_params_path} -- HPO not done yet")
        return

    if is_real:
        xes_path = ROOT / "data" / "real-life" / f"{dataset}.xes"
        df_full, cc, tt, train_df, val_df, test_df = load_data_for_ssd(xes_path)
    else:
        split = load_splits(dataset, "none", is_real=False)
        train_df, val_df, test_df = split["train"], split["val"], split["test"]
        cc, tt = split["cc"], split["tt"]

    winning = _load_camargo_winning_params(winning_trim_dir, winning_run_name)
    params = _build_fixed_camargo_params(f"{dataset}_seed{seed}", winning)

    set_global_seed(seed)
    tf.random.set_seed(seed)

    robustness_trim_dir = f"robustness_{sub}"
    robustness_run_name = f"{dataset}_seed{seed}"
    trainer = CamargoTrainer(train_df, val_df, test_df, robustness_run_name, params,
                             trim_dir=robustness_trim_dir)
    trainer.run()

    gen_paths = trainer.predict(full_prefix_only=True)
    event_log = trainer.to_event_log(gen_paths)

    from time_series_creation import create_concurrent_cases_timeseries, create_avg_throughtput_time_timeseries
    cc_pred = (create_concurrent_cases_timeseries(event_log, time_col="end_timestamp", case_col="caseid",
                                                  window="days", plot=False)
              .reindex(cc["test"].index).ffill().bfill().fillna(0))
    tt_pred = (create_avg_throughtput_time_timeseries(event_log, time_col="end_timestamp", case_col="caseid",
                                                       window="days", plot=False)
              .reindex(tt["test"].index).ffill().bfill().fillna(0))

    from sklearn.metrics import mean_absolute_error, mean_squared_error
    metrics = pd.DataFrame([
        dict(dataset=dataset, series="concurrent_cases", model="camargo", seed=seed,
             mae=mean_absolute_error(cc["test"], cc_pred), mse=mean_squared_error(cc["test"], cc_pred)),
        dict(dataset=dataset, series="throughput_time", model="camargo", seed=seed,
             mae=mean_absolute_error(tt["test"], tt_pred), mse=mean_squared_error(tt["test"], tt_pred)),
    ])

    out_dir.mkdir(parents=True, exist_ok=True)
    metrics.to_csv(metrics_path, index=False)
    event_log.to_csv(out_dir / "event_log.csv", index=False)

    src = GLSTM_OUT / robustness_trim_dir / robustness_run_name
    dst = out_dir / "model"
    if src.exists() and not dst.exists():
        shutil.copytree(src, dst)

    print(f"  [done] {dataset}/seed_{seed}: cc_mae={metrics.iloc[0]['mae']:.2f} tt_mae={metrics.iloc[1]['mae']:.2f}")

## Part 1: Synthetic

In [ ]:


for name in SYNTH_DATASETS:
    print(f"\n{'='*60}\n{name} (synth"
          f"etic)\n{'='*60}")
    for seed in NEW_SEEDS:
        run_camargo_robustness_one(name, is_real=False, seed=seed)

## Part 2: SSD

In [ ]:
def load_data_for_ssd(xes_path):
    """Returns (df, cc, tt, train_df, val_df, test_df) for the ssd trim."""
    from time_series_preprocessing import Split3WayConfig, split_timeseries
    from time_series_creation import create_concurrent_cases_timeseries, create_avg_throughtput_time_timeseries
    from create_prefixes_from_windows import make_three_way_split
    from ssd_trim import run_ssd_trim

    log = pm4py.read_xes(str(xes_path))

    full_cc_raw = create_concurrent_cases_timeseries(log, plot=False)
    ssd_result = run_ssd_trim(log, window_step="D")
    canonical_end = ssd_result["cutoff"] if ssd_result["cutoff"] is not None else full_cc_raw.index[-1]
    full_cc_trimmed = full_cc_raw[full_cc_raw.index <= canonical_end]
    split_cfg = Split3WayConfig(train_frac=0.70, val_frac=0.10, test_frac=0.20)
    _, _, _, train_split, val_split = split_timeseries(full_cc_trimmed, split_cfg)

    full_tt_raw = create_avg_throughtput_time_timeseries(log, plot=False)
    full_tt_trimmed = full_tt_raw[full_tt_raw.index <= canonical_end]

    def _slice(raw, trimmed):
        idx = trimmed.index
        lo = train_split.tz_convert(None) if idx.tz is None else train_split
        hi = val_split.tz_convert(None) if idx.tz is None else val_split
        return {
            "raw": raw, "trimmed": trimmed,
            "train": trimmed[idx <= lo],
            "val": trimmed[(idx > lo) & (idx <= hi)],
            "test": trimmed[idx > hi],
            "train_split": train_split, "val_split": val_split,
        }

    cc = _slice(full_cc_raw, full_cc_trimmed)
    tt = _slice(full_tt_raw, full_tt_trimmed)

    df = pm4py.convert_to_dataframe(log)
    df["time:timestamp"] = pd.to_datetime(df["time:timestamp"], utc=True)
    df = df.dropna(subset=["case:concept:name"])
    _cols = {"case:concept:name": "caseid", "concept:name": "task",
             "lifecycle:transition": "event_type", "time:timestamp": "end_timestamp"}
    _cols["org:resource" if "org:resource" in df.columns else "org:group"] = "user"
    df = df.rename(columns=_cols)
    df["task"] = df["task"].fillna("unk")
    df["user"] = df["user"].fillna("unk")

    train_, val_, test_ = make_three_way_split(
        df, case_col="caseid", time_col="end_timestamp",
        train_split=cc["train_split"], val_split=cc["val_split"], full_traces=True,
    )
    return df, cc, tt, train_, val_, test_

In [ ]:
for name in REAL_DATASETS:
    print(f"\n{'='*60}\n{name} (ssd)\n{'='*60}")
    for seed in NEW_SEEDS:
        run_camargo_robustness_one(name, is_real=True, seed=seed)

## Part 2 -- Bukhsh, new seeds


# Bukhsh robustness: retrain the HPO-selected best config with 2 new seeds

For each dataset and each seed in `NEW_SEEDS`: loads the winning hyperparameters, retrains from scratch, and saves both `bukhsh_suffix` and `bukhsh_rt` metrics.

Requires HPO to already be complete for that dataset.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import pm4py

ROOT = Path.cwd().resolve().parent.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "analysis"))
sys.path.insert(0, str(ROOT / "steady_state_detection"))

from setttings import set_global_seed
from ts_comparison import load_splits
from bukhsh.trainer import BukhshTrainer
from create_prefixes_from_windows import make_three_way_split
import tensorflow as tf

RESULTS = ROOT / "results"
BEST_MODELS = ROOT / "best_models"
ROBUST = ROOT / "robustness" / "bukhsh"

SYNTH_DATASETS = [p.stem for p in sorted((ROOT / "data" / "synthetic").glob("*.xes")) if "recency" not in p.stem]
REAL_DATASETS = ["bpic12-a", "bpic15-1", "bpic15-2", "bpic17-o",
                 "bpic20-dom", "bpic20-int", "helpdesk", "sepsis"]

NEW_SEEDS = [43, 44]

print(f"{len(SYNTH_DATASETS)} synthetic datasets, {len(REAL_DATASETS)} real-life (ssd) datasets, seeds={NEW_SEEDS}")

In [ ]:
def run_bukhsh_robustness_one(dataset: str, is_real: bool, seed: int):
    """One (dataset, seed) robustness run. Computes both bukhsh_suffix and bukhsh_rt metrics. Skips if metrics already exist."""
    sub = "ssd" if is_real else "synthetic"
    out_dir = ROBUST / sub / dataset / f"seed_{seed}"
    metrics_path = out_dir / "metrics.csv"
    if metrics_path.exists():
        print(f"  [skip] {dataset}/seed_{seed}: metrics already exist")
        return

    best_params_dir = (BEST_MODELS / dataset / "ssd" / "bukhsh") if is_real else (BEST_MODELS / dataset / "bukhsh")
    best_params_path = best_params_dir / "best_params.json"
    if not best_params_path.exists():
        print(f"  [skip] {dataset}: no trained bukhsh model at {best_params_path} -- HPO not done yet")
        return
    best_params = json.loads(best_params_path.read_text())

    if is_real:
        xes_path = ROOT / "data" / "real-life" / f"{dataset}.xes"
        df_full, cc, tt, train_df, val_df, test_df = load_data_for_ssd(xes_path)
    else:
        split = load_splits(dataset, "none", is_real=False)
        train_df, val_df, test_df = split["train"], split["val"], split["test"]
        cc, tt = split["cc"], split["tt"]

    set_global_seed(seed)
    tf.random.set_seed(seed)

    out_dir.mkdir(parents=True, exist_ok=True)
    trainer = BukhshTrainer(train_df, val_df, test_df, f"{dataset}_seed{seed}", best_params,
                            output_dir=out_dir / "model")
    trainer.run()
    suffix_log, rem_time_df = trainer.predict()

    from time_series_creation import create_concurrent_cases_timeseries, create_avg_throughtput_time_timeseries
    from run_predictions_real import _build_rt_log
    from sklearn.metrics import mean_absolute_error, mean_squared_error

    def _kpi(event_log):
        cc_p = (create_concurrent_cases_timeseries(event_log, time_col="end_timestamp", case_col="caseid",
                                                    window="days", plot=False)
               .reindex(cc["test"].index).ffill().bfill().fillna(0))
        tt_p = (create_avg_throughtput_time_timeseries(event_log, time_col="end_timestamp", case_col="caseid",
                                                        window="days", plot=False)
               .reindex(tt["test"].index).ffill().bfill().fillna(0))
        return cc_p, tt_p

    cc_pred_s, tt_pred_s = _kpi(suffix_log)

    rt_log = _build_rt_log(rem_time_df, test_df)
    cc_pred_rt, tt_pred_rt = _kpi(rt_log)

    metrics = pd.DataFrame([
        dict(dataset=dataset, series="concurrent_cases", model="bukhsh_suffix", seed=seed,
             mae=mean_absolute_error(cc["test"], cc_pred_s), mse=mean_squared_error(cc["test"], cc_pred_s)),
        dict(dataset=dataset, series="throughput_time", model="bukhsh_suffix", seed=seed,
             mae=mean_absolute_error(tt["test"], tt_pred_s), mse=mean_squared_error(tt["test"], tt_pred_s)),
        dict(dataset=dataset, series="concurrent_cases", model="bukhsh_rt", seed=seed,
             mae=mean_absolute_error(cc["test"], cc_pred_rt), mse=mean_squared_error(cc["test"], cc_pred_rt)),
        dict(dataset=dataset, series="throughput_time", model="bukhsh_rt", seed=seed,
             mae=mean_absolute_error(tt["test"], tt_pred_rt), mse=mean_squared_error(tt["test"], tt_pred_rt)),
    ])
    metrics.to_csv(metrics_path, index=False)
    suffix_log.to_csv(out_dir / "event_log.csv", index=False)
    rem_time_df.to_csv(out_dir / "rem_time.csv", index=False)

    print(f"  [done] {dataset}/seed_{seed}: "
         f"suffix cc_mae={metrics.iloc[0]['mae']:.2f} tt_mae={metrics.iloc[1]['mae']:.2f}  |  "
         f"rt cc_mae={metrics.iloc[2]['mae']:.2f} tt_mae={metrics.iloc[3]['mae']:.2f}")

## Part 1: Synthetic

In [ ]:
for name in SYNTH_DATASETS:
    print(f"\n{'='*60}\n{name} (synthetic)\n{'='*60}")
    for seed in NEW_SEEDS:
        run_bukhsh_robustness_one(name, is_real=False, seed=seed)

## Part 2: SSD

In [ ]:
def load_data_for_ssd(xes_path):
    """Returns (df, cc, tt, train_df, val_df, test_df) for the ssd trim."""
    from time_series_preprocessing import Split3WayConfig, split_timeseries
    from time_series_creation import create_concurrent_cases_timeseries, create_avg_throughtput_time_timeseries
    from create_prefixes_from_windows import make_three_way_split
    from ssd_trim import run_ssd_trim

    log = pm4py.read_xes(str(xes_path))

    full_cc_raw = create_concurrent_cases_timeseries(log, plot=False)
    ssd_result = run_ssd_trim(log, window_step="D")
    canonical_end = ssd_result["cutoff"] if ssd_result["cutoff"] is not None else full_cc_raw.index[-1]
    full_cc_trimmed = full_cc_raw[full_cc_raw.index <= canonical_end]
    split_cfg = Split3WayConfig(train_frac=0.70, val_frac=0.10, test_frac=0.20)
    _, _, _, train_split, val_split = split_timeseries(full_cc_trimmed, split_cfg)

    full_tt_raw = create_avg_throughtput_time_timeseries(log, plot=False)
    full_tt_trimmed = full_tt_raw[full_tt_raw.index <= canonical_end]

    def _slice(raw, trimmed):
        idx = trimmed.index
        lo = train_split.tz_convert(None) if idx.tz is None else train_split
        hi = val_split.tz_convert(None) if idx.tz is None else val_split
        return {
            "raw": raw, "trimmed": trimmed,
            "train": trimmed[idx <= lo],
            "val": trimmed[(idx > lo) & (idx <= hi)],
            "test": trimmed[idx > hi],
            "train_split": train_split, "val_split": val_split,
        }

    cc = _slice(full_cc_raw, full_cc_trimmed)
    tt = _slice(full_tt_raw, full_tt_trimmed)

    df = pm4py.convert_to_dataframe(log)
    df["time:timestamp"] = pd.to_datetime(df["time:timestamp"], utc=True)
    df = df.dropna(subset=["case:concept:name"])
    _cols = {"case:concept:name": "caseid", "concept:name": "task",
             "lifecycle:transition": "event_type", "time:timestamp": "end_timestamp"}
    _cols["org:resource" if "org:resource" in df.columns else "org:group"] = "user"
    df = df.rename(columns=_cols)
    df["task"] = df["task"].fillna("unk")
    df["user"] = df["user"].fillna("unk")

    train_, val_, test_ = make_three_way_split(
        df, case_col="caseid", time_col="end_timestamp",
        train_split=cc["train_split"], val_split=cc["val_split"], full_traces=True,
    )
    return df, cc, tt, train_, val_, test_

In [ ]:
for name in REAL_DATASETS:
    print(f"\n{'='*60}\n{name} (ssd)\n{'='*60}")
    for seed in NEW_SEEDS:
        run_bukhsh_robustness_one(name, is_real=True, seed=seed)

## Part 3 -- Amiri, new seeds


# Amiri robustness: retrain the HPO-selected best config with 2 new seeds

For each dataset and each seed in `NEW_SEEDS`: loads the winning hyperparameters, retrains from scratch, and saves metrics.

Requires HPO to already be complete for that dataset.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import pm4py

ROOT = Path.cwd().resolve().parent.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "analysis"))
sys.path.insert(0, str(ROOT / "steady_state_detection"))

from setttings import set_global_seed
from ts_comparison import load_splits
from amiri.trainer import AmiriTrainer

RESULTS = ROOT / "results"
BEST_MODELS = ROOT / "best_models"
ROBUST = ROOT / "robustness" / "amiri"

SYNTH_DATASETS = [p.stem for p in sorted((ROOT / "data" / "synthetic").glob("*.xes")) if "recency" not in p.stem]
REAL_DATASETS = ["bpic12-a", "bpic15-1", "bpic15-2", "bpic17-o",
                 "bpic20-dom", "bpic20-int", "helpdesk", "sepsis"]

NEW_SEEDS = [43, 44]

print(f"{len(SYNTH_DATASETS)} synthetic datasets, {len(REAL_DATASETS)} real-life (ssd) datasets, seeds={NEW_SEEDS}")

In [ ]:
def rem_time_to_event_log(rt_df):
    rt = rt_df.copy()
    rt["start_timestamp"] = pd.to_datetime(rt["start_timestamp"])
    rt["anchor_timestamp"] = pd.to_datetime(rt["anchor_timestamp"])
    rt["predicted_end"] = rt["anchor_timestamp"] + pd.to_timedelta(rt["rem_time_days"], unit="D")
    return pd.concat([
        rt[["caseid", "start_timestamp"]].rename(columns={"start_timestamp": "end_timestamp"}),
        rt[["caseid", "predicted_end"]].rename(columns={"predicted_end": "end_timestamp"}),
    ], ignore_index=True)


def run_amiri_robustness_one(dataset: str, is_real: bool, seed: int):
    """One (dataset, seed) robustness run. Skips if metrics already exist."""
    sub = "ssd" if is_real else "synthetic"
    out_dir = ROBUST / sub / dataset / f"seed_{seed}"
    metrics_path = out_dir / "metrics.csv"
    if metrics_path.exists():
        print(f"  [skip] {dataset}/seed_{seed}: metrics already exist")
        return

    best_params_dir = ((BEST_MODELS / dataset / "ssd" / "amiri" / f"{dataset}_full") if is_real
                       else (BEST_MODELS / dataset / "amiri" / "none" / f"{dataset}_full"))
    best_params_path = best_params_dir / "best_params.json"
    if not best_params_path.exists():
        print(f"  [skip] {dataset}: no trained amiri model at {best_params_path} -- HPO not done yet")
        return
    params = json.loads(best_params_path.read_text())
    params["seed"] = seed

    if is_real:
        xes_path = ROOT / "data" / "real-life" / f"{dataset}.xes"
        df_full, cc, tt, train_df, val_df, test_df = load_data_for_ssd(xes_path)
    else:
        split = load_splits(dataset, "none", is_real=False)
        train_df, val_df, test_df = split["train"], split["val"], split["test"]
        cc, tt = split["cc"], split["tt"]

    out_dir.mkdir(parents=True, exist_ok=True)
    trainer = AmiriTrainer(train_df, val_df, test_df, run_name=f"{dataset}_seed{seed}",
                           params=params, output_dir=out_dir / "model",
                           dataset_dir=out_dir / "model" / "dataset")
    trainer.run()
    rem_time_df = trainer.predict()
    event_log = rem_time_to_event_log(rem_time_df)

    from time_series_creation import create_concurrent_cases_timeseries, create_avg_throughtput_time_timeseries
    cc_pred = (create_concurrent_cases_timeseries(event_log, time_col="end_timestamp", case_col="caseid",
                                                  window="days", plot=False)
              .reindex(cc["test"].index).ffill().bfill().fillna(0))
    tt_pred = (create_avg_throughtput_time_timeseries(event_log, time_col="end_timestamp", case_col="caseid",
                                                       window="days", plot=False)
              .reindex(tt["test"].index).ffill().bfill().fillna(0))

    from sklearn.metrics import mean_absolute_error, mean_squared_error
    metrics = pd.DataFrame([
        dict(dataset=dataset, series="concurrent_cases", model="amiri", seed=seed,
             mae=mean_absolute_error(cc["test"], cc_pred), mse=mean_squared_error(cc["test"], cc_pred)),
        dict(dataset=dataset, series="throughput_time", model="amiri", seed=seed,
             mae=mean_absolute_error(tt["test"], tt_pred), mse=mean_squared_error(tt["test"], tt_pred)),
    ])
    metrics.to_csv(metrics_path, index=False)
    rem_time_df.to_csv(out_dir / "rem_time.csv", index=False)

    print(f"  [done] {dataset}/seed_{seed}: cc_mae={metrics.iloc[0]['mae']:.2f} tt_mae={metrics.iloc[1]['mae']:.2f}")

## Part 1: Synthetic

In [ ]:
for name in SYNTH_DATASETS:
    print(f"\n{'='*60}\n{name} (synthetic)\n{'='*60}")
    for seed in NEW_SEEDS:
        run_amiri_robustness_one(name, is_real=False, seed=seed)

## Part 2: SSD

In [ ]:
def load_data_for_ssd(xes_path):
    """Returns (df, cc, tt, train_df, val_df, test_df) for the ssd trim."""
    from time_series_preprocessing import Split3WayConfig, split_timeseries
    from time_series_creation import create_concurrent_cases_timeseries, create_avg_throughtput_time_timeseries
    from create_prefixes_from_windows import make_three_way_split
    from ssd_trim import run_ssd_trim

    log = pm4py.read_xes(str(xes_path))

    full_cc_raw = create_concurrent_cases_timeseries(log, plot=False)
    ssd_result = run_ssd_trim(log, window_step="D")
    canonical_end = ssd_result["cutoff"] if ssd_result["cutoff"] is not None else full_cc_raw.index[-1]
    full_cc_trimmed = full_cc_raw[full_cc_raw.index <= canonical_end]
    split_cfg = Split3WayConfig(train_frac=0.70, val_frac=0.10, test_frac=0.20)
    _, _, _, train_split, val_split = split_timeseries(full_cc_trimmed, split_cfg)

    full_tt_raw = create_avg_throughtput_time_timeseries(log, plot=False)
    full_tt_trimmed = full_tt_raw[full_tt_raw.index <= canonical_end]

    def _slice(raw, trimmed):
        idx = trimmed.index
        lo = train_split.tz_convert(None) if idx.tz is None else train_split
        hi = val_split.tz_convert(None) if idx.tz is None else val_split
        return {
            "raw": raw, "trimmed": trimmed,
            "train": trimmed[idx <= lo],
            "val": trimmed[(idx > lo) & (idx <= hi)],
            "test": trimmed[idx > hi],
            "train_split": train_split, "val_split": val_split,
        }

    cc = _slice(full_cc_raw, full_cc_trimmed)
    tt = _slice(full_tt_raw, full_tt_trimmed)

    df = pm4py.convert_to_dataframe(log)
    df["time:timestamp"] = pd.to_datetime(df["time:timestamp"], utc=True)
    df = df.dropna(subset=["case:concept:name"])
    _cols = {"case:concept:name": "caseid", "concept:name": "task",
             "lifecycle:transition": "event_type", "time:timestamp": "end_timestamp"}
    _cols["org:resource" if "org:resource" in df.columns else "org:group"] = "user"
    df = df.rename(columns=_cols)
    df["task"] = df["task"].fillna("unk")
    df["user"] = df["user"].fillna("unk")

    train_, val_, test_ = make_three_way_split(
        df, case_col="caseid", time_col="end_timestamp",
        train_split=cc["train_split"], val_split=cc["val_split"], full_traces=True,
    )
    return df, cc, tt, train_, val_, test_

In [ ]:
for name in REAL_DATASETS:
    print(f"\n{'='*60}\n{name} (ssd)\n{'='*60}")
    for seed in NEW_SEEDS:
        run_amiri_robustness_one(name, is_real=True, seed=seed)

## Part 4 -- Half-prefix + plain-field, all three models, both new seeds

# Half-prefix + plain-field robustness: camargo, bukhsh, amiri (synthetic + ssd)

Prediction only -- reuses each `(approach, dataset, seed)`'s already-retrained full-regime model. Requires that full regime to already exist. Output: `half/metrics.csv` and `pf/metrics.csv` under each existing `seed_<N>/` directory.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import pm4py
import tensorflow as tf

ROOT = Path.cwd().resolve().parent.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "analysis"))
sys.path.insert(0, str(ROOT / "steady_state_detection"))
sys.path.insert(0, str(ROOT / "GenerativeLSTM" / "GenerativeLSTM"))
sys.path.insert(0, str(ROOT / "plain-field"))

from setttings import set_global_seed
from ts_comparison import load_splits
from run_predictions_real import _make_half_prefix_test_df, _build_rt_log

from camargo.trainer import CamargoTrainer
from camargo.params import default_params as camargo_default_params
from bukhsh.trainer import BukhshTrainer
from amiri.trainer import AmiriTrainer

from runner import (
    get_inflight_cases, build_sos_cases,
    predict_camargo_plain_field, predict_bukhsh_plain_field, predict_amiri_plain_field,
    compute_cc_tt_metrics,
)
from sos import most_frequent_first_activity, most_frequent_first_resource, empirical_arrival_hour_sampler
from arrival import compute_arrival_series, ProphetArrivalModel

RESULTS = ROOT / "results"
BEST_MODELS = ROOT / "best_models"
GLSTM_OUT = ROOT / "GenerativeLSTM" / "GenerativeLSTM" / "output_files"
ROBUST_ROOT = ROOT / "robustness"

SYNTH_DATASETS = [p.stem for p in sorted((ROOT / "data" / "synthetic").glob("*.xes")) if "recency" not in p.stem]
REAL_DATASETS = ["bpic12-a", "bpic15-1", "bpic15-2", "bpic17-o",
                 "bpic20-dom", "bpic20-int", "helpdesk", "sepsis"]
NEW_SEEDS = [43, 44]

print(f"{len(SYNTH_DATASETS)} synthetic datasets, {len(REAL_DATASETS)} real-life (ssd) datasets, seeds={NEW_SEEDS}")


def load_data_for_ssd(xes_path):
    """Returns (df, cc, tt, train_df, val_df, test_df) for the ssd trim."""
    from time_series_preprocessing import Split3WayConfig, split_timeseries
    from time_series_creation import create_concurrent_cases_timeseries, create_avg_throughtput_time_timeseries
    from create_prefixes_from_windows import make_three_way_split
    from ssd_trim import run_ssd_trim

    log = pm4py.read_xes(str(xes_path))

    full_cc_raw = create_concurrent_cases_timeseries(log, plot=False)
    ssd_result = run_ssd_trim(log, window_step="D")
    canonical_end = ssd_result["cutoff"] if ssd_result["cutoff"] is not None else full_cc_raw.index[-1]
    full_cc_trimmed = full_cc_raw[full_cc_raw.index <= canonical_end]
    split_cfg = Split3WayConfig(train_frac=0.70, val_frac=0.10, test_frac=0.20)
    _, _, _, train_split, val_split = split_timeseries(full_cc_trimmed, split_cfg)

    full_tt_raw = create_avg_throughtput_time_timeseries(log, plot=False)
    full_tt_trimmed = full_tt_raw[full_tt_raw.index <= canonical_end]

    def _slice(raw, trimmed):
        idx = trimmed.index
        lo = train_split.tz_convert(None) if idx.tz is None else train_split
        hi = val_split.tz_convert(None) if idx.tz is None else val_split
        return {
            "raw": raw, "trimmed": trimmed,
            "train": trimmed[idx <= lo],
            "val": trimmed[(idx > lo) & (idx <= hi)],
            "test": trimmed[idx > hi],
            "train_split": train_split, "val_split": val_split,
        }

    cc = _slice(full_cc_raw, full_cc_trimmed)
    tt = _slice(full_tt_raw, full_tt_trimmed)

    df = pm4py.convert_to_dataframe(log)
    df["time:timestamp"] = pd.to_datetime(df["time:timestamp"], utc=True)
    df = df.dropna(subset=["case:concept:name"])
    _cols = {"case:concept:name": "caseid", "concept:name": "task",
             "lifecycle:transition": "event_type", "time:timestamp": "end_timestamp"}
    _cols["org:resource" if "org:resource" in df.columns else "org:group"] = "user"
    df = df.rename(columns=_cols)
    df["task"] = df["task"].fillna("unk")
    df["user"] = df["user"].fillna("unk")

    train_, val_, test_ = make_three_way_split(
        df, case_col="caseid", time_col="end_timestamp",
        train_split=cc["train_split"], val_split=cc["val_split"], full_traces=True,
    )
    return df, cc, tt, train_, val_, test_


def _load_split(dataset: str, is_real: bool):
    """Returns (df_full, cc, tt, train_df, val_df, test_df) -- shared by every
    section below."""
    if is_real:
        xes_path = ROOT / "data" / "real-life" / f"{dataset}.xes"
        return load_data_for_ssd(xes_path)
    split = load_splits(dataset, "none", is_real=False)
    return (split["df"], split["cc"], split["tt"], split["train"], split["val"], split["test"])


## Camargo -- half-prefix and plain-field

In [ ]:
_CAMARGO_DROPOUT = 0.2
_CAMARGO_LEARNING_RATE = 0.002


def _load_camargo_winning_params(trim_dir: str, run_name: str) -> dict:
    """Loads the winning trial's architecture params, filling dropout/learning_rate from a fixed default when absent."""
    p = GLSTM_OUT / trim_dir / run_name / "parameters" / "model_parameters.json"
    saved = json.loads(p.read_text())
    return {
        "model_type": saved["model_type"], "lstm_act": saved["lstm_act"],
        "dense_act": saved["dense_act"], "n_size": saved["n_size"],
        "l_size": saved["l_size"], "norm_method": saved["norm_method"],
        "optim": saved["optim"], "dropout": saved.get("dropout", _CAMARGO_DROPOUT),
        "learning_rate": saved.get("learning_rate", _CAMARGO_LEARNING_RATE),
    }


def _build_fixed_camargo_params(run_name: str, winning: dict) -> dict:
    """default_params() with every HPO-searched key pinned to a single value, max_eval=1."""
    p = camargo_default_params(run_name, max_eval=1, epochs=200)
    p["model_type"] = [winning["model_type"]]
    p["lstm_act"] = [winning["lstm_act"]]
    p["dense_act"] = [winning["dense_act"]]
    p["n_size"] = [winning["n_size"]]
    p["l_size"] = [winning["l_size"]]
    p["norm_method"] = [winning["norm_method"]]
    p["optim"] = [winning["optim"]]
    p["dropout"] = [winning["dropout"]]
    p["learning_rate"] = [winning["learning_rate"]]
    return p


def _camargo_reload(dataset: str, sub: str, seed: int, test_df_for_predict):
    """Reloads the already-trained model for this (dataset, seed)."""
    is_real = sub == "ssd"
    winning_run_name = f"{dataset}_ssd_hpo" if is_real else f"{dataset}_hpo"
    winning_trim_dir = "ssd" if is_real else "hpo"
    winning = _load_camargo_winning_params(winning_trim_dir, winning_run_name)
    params = _build_fixed_camargo_params(f"{dataset}_seed{seed}", winning)

    df_full, cc, tt, train_df, val_df, _ = _load_split(dataset, is_real)

    robustness_trim_dir = f"robustness_{sub}"
    robustness_run_name = f"{dataset}_seed{seed}"
    tc = CamargoTrainer.from_saved(train_df, val_df, test_df_for_predict, robustness_run_name, params,
                                   trim_dir=robustness_trim_dir)
    return tc, df_full, cc, tt, train_df, val_df


def run_camargo_half(dataset: str, is_real: bool, seed: int):
    sub = "ssd" if is_real else "synthetic"
    out_dir = ROBUST_ROOT / "camargo" / sub / dataset / f"seed_{seed}"
    half_dir = out_dir / "half"
    metrics_path = half_dir / "metrics.csv"
    if metrics_path.exists():
        print(f"  [skip] {dataset}/seed_{seed}/half: metrics already exist")
        return
    if not (out_dir / "metrics.csv").exists():
        print(f"  [skip] {dataset}/seed_{seed}/half: full regime hasn't run yet")
        return

    df_full, cc, tt, train_df, val_df, test_df = _load_split(dataset, is_real)
    test_df_half = _make_half_prefix_test_df(df_full, test_df)
    tc, *_ = _camargo_reload(dataset, sub, seed, test_df_half)

    import shutil
    orig_name, orig_out = tc.run_name, tc.output_dir
    half_probe_name = orig_name + "_half"
    orig_out_abs = GLSTM_OUT / orig_out
    half_out_abs = orig_out_abs.parent / half_probe_name
    half_params_dir = half_out_abs / "parameters"
    half_params_dir.mkdir(parents=True, exist_ok=True)
    orig_params_dir = orig_out_abs / "parameters"
    for fname in ("model_parameters.json", "resource_map.csv"):
        src = orig_params_dir / fname
        if src.exists():
            shutil.copy2(src, half_params_dir / fname)
    for h5 in orig_out_abs.glob("*.h5"):
        shutil.copy2(h5, half_out_abs / h5.name)
    tc.run_name = half_probe_name
    tc.output_dir = str(Path(orig_out).parent / half_probe_name)
    try:
        gen_paths = tc.predict(full_prefix_only=True)
        event_log = tc.to_event_log(gen_paths)
    finally:
        tc.run_name, tc.output_dir = orig_name, orig_out

    from time_series_creation import create_concurrent_cases_timeseries, create_avg_throughtput_time_timeseries
    from sklearn.metrics import mean_absolute_error, mean_squared_error
    cc_pred = (create_concurrent_cases_timeseries(event_log, time_col="end_timestamp", case_col="caseid",
                                                  window="days", plot=False)
              .reindex(cc["test"].index).ffill().bfill().fillna(0))
    tt_pred = (create_avg_throughtput_time_timeseries(event_log, time_col="end_timestamp", case_col="caseid",
                                                       window="days", plot=False)
              .reindex(tt["test"].index).ffill().bfill().fillna(0))

    metrics = pd.DataFrame([
        dict(dataset=dataset, series="concurrent_cases", model="camargo", seed=seed,
             mae=mean_absolute_error(cc["test"], cc_pred), mse=mean_squared_error(cc["test"], cc_pred)),
        dict(dataset=dataset, series="throughput_time", model="camargo", seed=seed,
             mae=mean_absolute_error(tt["test"], tt_pred), mse=mean_squared_error(tt["test"], tt_pred)),
    ])
    half_dir.mkdir(parents=True, exist_ok=True)
    metrics.to_csv(metrics_path, index=False)
    event_log.to_csv(half_dir / "event_log.csv", index=False)
    print(f"  [done] {dataset}/seed_{seed}/half: cc_mae={metrics.iloc[0]['mae']:.2f} tt_mae={metrics.iloc[1]['mae']:.2f}")


def run_camargo_pf(dataset: str, is_real: bool, seed: int):
    sub = "ssd" if is_real else "synthetic"
    out_dir = ROBUST_ROOT / "camargo" / sub / dataset / f"seed_{seed}"
    pf_dir = out_dir / "pf"
    metrics_path = pf_dir / "metrics.csv"
    if metrics_path.exists():
        print(f"  [skip] {dataset}/seed_{seed}/pf: metrics already exist")
        return
    if not (out_dir / "metrics.csv").exists():
        print(f"  [skip] {dataset}/seed_{seed}/pf: full regime hasn't run yet")
        return

    df_full, cc, tt, train_df, val_df, test_df = _load_split(dataset, is_real)

    val_split = cc["val_split"]
    inflight_df = get_inflight_cases(df_full, val_split, case_col="caseid", time_col="end_timestamp")
    known_df = pd.concat([train_df, val_df], ignore_index=True)
    val_split_ts = pd.Timestamp(val_split)
    if val_split_ts.tzinfo:
        val_split_ts = val_split_ts.tz_convert(None)
    arrivals = compute_arrival_series(df_full, case_col="caseid", time_col="end_timestamp")
    arrivals = arrivals[arrivals.index < val_split_ts]
    arrival_model = ProphetArrivalModel().fit(arrivals)
    predicted_arrivals = arrival_model.predict(pd.DatetimeIndex(cc["test"].index).tz_localize(None))
    sos_df = build_sos_cases(
        predicted_arrivals,
        most_frequent_first_activity(known_df), most_frequent_first_resource(known_df),
        empirical_arrival_hour_sampler(known_df),
    )

    tc, *_ = _camargo_reload(dataset, sub, seed, test_df)
    pred_log = predict_camargo_plain_field(tc, sos_df, inflight_df)

    cc_p, tt_p, m = compute_cc_tt_metrics(pred_log, cc["test"], tt["test"])
    metrics = pd.DataFrame([
        dict(dataset=dataset, series="concurrent_cases", model="camargo", seed=seed, mae=m["cc_mae"], mse=m["cc_mse"]),
        dict(dataset=dataset, series="throughput_time", model="camargo", seed=seed, mae=m["tt_mae"], mse=m["tt_mse"]),
    ])
    pf_dir.mkdir(parents=True, exist_ok=True)
    metrics.to_csv(metrics_path, index=False)
    pred_log.to_csv(pf_dir / "event_log.csv", index=False)
    print(f"  [done] {dataset}/seed_{seed}/pf: cc_mae={m['cc_mae']:.2f} tt_mae={m['tt_mae']:.2f}")


for name in SYNTH_DATASETS:
    print(f"\n{'='*60}\n{name} (camargo, synthetic, half)\n{'='*60}")
    for seed in NEW_SEEDS:
        run_camargo_half(name, is_real=False, seed=seed)
for name in REAL_DATASETS:
    print(f"\n{'='*60}\n{name} (camargo, ssd, half)\n{'='*60}")
    for seed in NEW_SEEDS:
        run_camargo_half(name, is_real=True, seed=seed)

for name in SYNTH_DATASETS:
    print(f"\n{'='*60}\n{name} (camargo, synthetic, pf)\n{'='*60}")
    for seed in NEW_SEEDS:
        run_camargo_pf(name, is_real=False, seed=seed)
for name in REAL_DATASETS:
    print(f"\n{'='*60}\n{name} (camargo, ssd, pf)\n{'='*60}")
    for seed in NEW_SEEDS:
        run_camargo_pf(name, is_real=True, seed=seed)


## Bukhsh -- half-prefix and plain-field

In [ ]:
def _bukhsh_kpi(event_log, cc, tt):
    from time_series_creation import create_concurrent_cases_timeseries, create_avg_throughtput_time_timeseries
    cc_p = (create_concurrent_cases_timeseries(event_log, time_col="end_timestamp", case_col="caseid",
                                                window="days", plot=False)
           .reindex(cc["test"].index).ffill().bfill().fillna(0))
    tt_p = (create_avg_throughtput_time_timeseries(event_log, time_col="end_timestamp", case_col="caseid",
                                                    window="days", plot=False)
           .reindex(tt["test"].index).ffill().bfill().fillna(0))
    return cc_p, tt_p


def _bukhsh_metrics_df(dataset, seed, cc, tt, cc_pred_s, tt_pred_s, cc_pred_rt, tt_pred_rt):
    from sklearn.metrics import mean_absolute_error, mean_squared_error
    return pd.DataFrame([
        dict(dataset=dataset, series="concurrent_cases", model="bukhsh_suffix", seed=seed,
             mae=mean_absolute_error(cc["test"], cc_pred_s), mse=mean_squared_error(cc["test"], cc_pred_s)),
        dict(dataset=dataset, series="throughput_time", model="bukhsh_suffix", seed=seed,
             mae=mean_absolute_error(tt["test"], tt_pred_s), mse=mean_squared_error(tt["test"], tt_pred_s)),
        dict(dataset=dataset, series="concurrent_cases", model="bukhsh_rt", seed=seed,
             mae=mean_absolute_error(cc["test"], cc_pred_rt), mse=mean_squared_error(cc["test"], cc_pred_rt)),
        dict(dataset=dataset, series="throughput_time", model="bukhsh_rt", seed=seed,
             mae=mean_absolute_error(tt["test"], tt_pred_rt), mse=mean_squared_error(tt["test"], tt_pred_rt)),
    ])


def run_bukhsh_half(dataset: str, is_real: bool, seed: int):
    sub = "ssd" if is_real else "synthetic"
    out_dir = ROBUST_ROOT / "bukhsh" / sub / dataset / f"seed_{seed}"
    half_dir = out_dir / "half"
    metrics_path = half_dir / "metrics.csv"
    if metrics_path.exists():
        print(f"  [skip] {dataset}/seed_{seed}/half: metrics already exist")
        return
    model_dir = out_dir / "model"
    if not model_dir.exists():
        print(f"  [skip] {dataset}/seed_{seed}/half: full regime hasn't run yet")
        return

    best_params_dir = (BEST_MODELS / dataset / "ssd" / "bukhsh") if is_real else (BEST_MODELS / dataset / "bukhsh")
    best_params = json.loads((best_params_dir / "best_params.json").read_text())

    df_full, cc, tt, train_df, val_df, test_df = _load_split(dataset, is_real)
    test_df_half = _make_half_prefix_test_df(df_full, test_df)

    trainer = BukhshTrainer(train_df, val_df, test_df_half, f"{dataset}_seed{seed}", best_params,
                            output_dir=model_dir)
    suffix_log, rem_time_df = trainer.predict()

    cc_pred_s, tt_pred_s = _bukhsh_kpi(suffix_log, cc, tt)
    rt_log = _build_rt_log(rem_time_df, test_df_half)
    cc_pred_rt, tt_pred_rt = _bukhsh_kpi(rt_log, cc, tt)

    metrics = _bukhsh_metrics_df(dataset, seed, cc, tt, cc_pred_s, tt_pred_s, cc_pred_rt, tt_pred_rt)
    half_dir.mkdir(parents=True, exist_ok=True)
    metrics.to_csv(metrics_path, index=False)
    suffix_log.to_csv(half_dir / "event_log.csv", index=False)
    rem_time_df.to_csv(half_dir / "rem_time.csv", index=False)
    print(f"  [done] {dataset}/seed_{seed}/half: suffix cc_mae={metrics.iloc[0]['mae']:.2f}  rt cc_mae={metrics.iloc[2]['mae']:.2f}")


def run_bukhsh_pf(dataset: str, is_real: bool, seed: int):
    sub = "ssd" if is_real else "synthetic"
    out_dir = ROBUST_ROOT / "bukhsh" / sub / dataset / f"seed_{seed}"
    pf_dir = out_dir / "pf"
    metrics_path = pf_dir / "metrics.csv"
    if metrics_path.exists():
        print(f"  [skip] {dataset}/seed_{seed}/pf: metrics already exist")
        return
    model_dir = out_dir / "model"
    if not model_dir.exists():
        print(f"  [skip] {dataset}/seed_{seed}/pf: full regime hasn't run yet")
        return

    best_params_dir = (BEST_MODELS / dataset / "ssd" / "bukhsh") if is_real else (BEST_MODELS / dataset / "bukhsh")
    best_params = json.loads((best_params_dir / "best_params.json").read_text())

    df_full, cc, tt, train_df, val_df, test_df = _load_split(dataset, is_real)

    val_split = cc["val_split"]
    inflight_df = get_inflight_cases(df_full, val_split, case_col="caseid", time_col="end_timestamp")
    known_df = pd.concat([train_df, val_df], ignore_index=True)
    val_split_ts = pd.Timestamp(val_split)
    if val_split_ts.tzinfo:
        val_split_ts = val_split_ts.tz_convert(None)
    arrivals = compute_arrival_series(df_full, case_col="caseid", time_col="end_timestamp")
    arrivals = arrivals[arrivals.index < val_split_ts]
    arrival_model = ProphetArrivalModel().fit(arrivals)
    predicted_arrivals = arrival_model.predict(pd.DatetimeIndex(cc["test"].index).tz_localize(None))
    sos_df = build_sos_cases(
        predicted_arrivals,
        most_frequent_first_activity(known_df), most_frequent_first_resource(known_df),
        empirical_arrival_hour_sampler(known_df),
    )

    trainer = BukhshTrainer(train_df, val_df, test_df, f"{dataset}_seed{seed}", best_params,
                            output_dir=model_dir)
    rem_time_df, suffix_log = predict_bukhsh_plain_field(trainer, sos_df, inflight_df)

    cc_p_s, tt_p_s, m_s = compute_cc_tt_metrics(suffix_log, cc["test"], tt["test"])
    rt_log = _build_rt_log(rem_time_df, test_df)
    cc_p_rt, tt_p_rt, m_rt = compute_cc_tt_metrics(rt_log, cc["test"], tt["test"])

    metrics = pd.DataFrame([
        dict(dataset=dataset, series="concurrent_cases", model="bukhsh_suffix", seed=seed, mae=m_s["cc_mae"], mse=m_s["cc_mse"]),
        dict(dataset=dataset, series="throughput_time", model="bukhsh_suffix", seed=seed, mae=m_s["tt_mae"], mse=m_s["tt_mse"]),
        dict(dataset=dataset, series="concurrent_cases", model="bukhsh_rt", seed=seed, mae=m_rt["cc_mae"], mse=m_rt["cc_mse"]),
        dict(dataset=dataset, series="throughput_time", model="bukhsh_rt", seed=seed, mae=m_rt["tt_mae"], mse=m_rt["tt_mse"]),
    ])
    pf_dir.mkdir(parents=True, exist_ok=True)
    metrics.to_csv(metrics_path, index=False)
    suffix_log.to_csv(pf_dir / "event_log.csv", index=False)
    rem_time_df.to_csv(pf_dir / "rem_time.csv", index=False)
    print(f"  [done] {dataset}/seed_{seed}/pf: suffix cc_mae={m_s['cc_mae']:.2f}  rt cc_mae={m_rt['cc_mae']:.2f}")


for name in SYNTH_DATASETS:
    print(f"\n{'='*60}\n{name} (bukhsh, synthetic, half)\n{'='*60}")
    for seed in NEW_SEEDS:
        run_bukhsh_half(name, is_real=False, seed=seed)
for name in REAL_DATASETS:
    print(f"\n{'='*60}\n{name} (bukhsh, ssd, half)\n{'='*60}")
    for seed in NEW_SEEDS:
        run_bukhsh_half(name, is_real=True, seed=seed)

for name in SYNTH_DATASETS:
    print(f"\n{'='*60}\n{name} (bukhsh, synthetic, pf)\n{'='*60}")
    for seed in NEW_SEEDS:
        run_bukhsh_pf(name, is_real=False, seed=seed)
for name in REAL_DATASETS:
    print(f"\n{'='*60}\n{name} (bukhsh, ssd, pf)\n{'='*60}")
    for seed in NEW_SEEDS:
        run_bukhsh_pf(name, is_real=True, seed=seed)


## Amiri -- half-prefix and plain-field

In [ ]:
def rem_time_to_event_log(rt_df):
    rt = rt_df.copy()
    rt["start_timestamp"] = pd.to_datetime(rt["start_timestamp"])
    rt["anchor_timestamp"] = pd.to_datetime(rt["anchor_timestamp"])
    rt["predicted_end"] = rt["anchor_timestamp"] + pd.to_timedelta(rt["rem_time_days"], unit="D")
    return pd.concat([
        rt[["caseid", "start_timestamp"]].rename(columns={"start_timestamp": "end_timestamp"}),
        rt[["caseid", "predicted_end"]].rename(columns={"predicted_end": "end_timestamp"}),
    ], ignore_index=True)


def _amiri_best_params(dataset: str, is_real: bool, seed: int) -> dict:
    best_params_dir = ((BEST_MODELS / dataset / "ssd" / "amiri" / f"{dataset}_full") if is_real
                       else (BEST_MODELS / dataset / "amiri" / "none" / f"{dataset}_full"))
    params = json.loads((best_params_dir / "best_params.json").read_text())
    params["seed"] = seed
    return params


def run_amiri_half(dataset: str, is_real: bool, seed: int):
    sub = "ssd" if is_real else "synthetic"
    out_dir = ROBUST_ROOT / "amiri" / sub / dataset / f"seed_{seed}"
    half_dir = out_dir / "half"
    metrics_path = half_dir / "metrics.csv"
    if metrics_path.exists():
        print(f"  [skip] {dataset}/seed_{seed}/half: metrics already exist")
        return

    model_dir = out_dir / "model"
    dataset_dir = model_dir / "dataset"
    if not (model_dir / "gps_results").exists():
        print(f"  [skip] {dataset}/seed_{seed}/half: full regime hasn't run yet")
        return

    params = _amiri_best_params(dataset, is_real, seed)
    df_full, cc, tt, train_df, val_df, test_df = _load_split(dataset, is_real)
    test_df_half = _make_half_prefix_test_df(df_full, test_df)

    half_dir.mkdir(parents=True, exist_ok=True)
    half_dataset_dir = half_dir / "dataset"
    gps_link = half_dir / "gps_results"
    if gps_link.exists() or gps_link.is_symlink():
        gps_link.unlink()
    gps_link.symlink_to((model_dir / "gps_results").resolve())

    trainer_half = AmiriTrainer(train_df, val_df, test_df_half, run_name=f"{dataset}_seed{seed}",
                                params=params, output_dir=half_dir,
                                dataset_dir=half_dataset_dir, ref_dataset_dir=dataset_dir)
    rem_time_df = trainer_half.predict()
    event_log = rem_time_to_event_log(rem_time_df)

    from time_series_creation import create_concurrent_cases_timeseries, create_avg_throughtput_time_timeseries
    from sklearn.metrics import mean_absolute_error, mean_squared_error
    cc_pred = (create_concurrent_cases_timeseries(event_log, time_col="end_timestamp", case_col="caseid",
                                                  window="days", plot=False)
              .reindex(cc["test"].index).ffill().bfill().fillna(0))
    tt_pred = (create_avg_throughtput_time_timeseries(event_log, time_col="end_timestamp", case_col="caseid",
                                                       window="days", plot=False)
              .reindex(tt["test"].index).ffill().bfill().fillna(0))

    metrics = pd.DataFrame([
        dict(dataset=dataset, series="concurrent_cases", model="amiri", seed=seed,
             mae=mean_absolute_error(cc["test"], cc_pred), mse=mean_squared_error(cc["test"], cc_pred)),
        dict(dataset=dataset, series="throughput_time", model="amiri", seed=seed,
             mae=mean_absolute_error(tt["test"], tt_pred), mse=mean_squared_error(tt["test"], tt_pred)),
    ])
    metrics.to_csv(metrics_path, index=False)
    rem_time_df.to_csv(half_dir / "rem_time.csv", index=False)
    print(f"  [done] {dataset}/seed_{seed}/half: cc_mae={metrics.iloc[0]['mae']:.2f} tt_mae={metrics.iloc[1]['mae']:.2f}")


def run_amiri_pf(dataset: str, is_real: bool, seed: int):
    sub = "ssd" if is_real else "synthetic"
    out_dir = ROBUST_ROOT / "amiri" / sub / dataset / f"seed_{seed}"
    pf_dir = out_dir / "pf"
    metrics_path = pf_dir / "metrics.csv"
    if metrics_path.exists():
        print(f"  [skip] {dataset}/seed_{seed}/pf: metrics already exist")
        return

    model_dir = out_dir / "model"
    dataset_dir = model_dir / "dataset"
    if not (model_dir / "gps_results").exists():
        print(f"  [skip] {dataset}/seed_{seed}/pf: full regime hasn't run yet")
        return

    params = _amiri_best_params(dataset, is_real, seed)
    df_full, cc, tt, train_df, val_df, test_df = _load_split(dataset, is_real)

    val_split = cc["val_split"]
    inflight_df = get_inflight_cases(df_full, val_split, case_col="caseid", time_col="end_timestamp")
    known_df = pd.concat([train_df, val_df], ignore_index=True)
    val_split_ts = pd.Timestamp(val_split)
    if val_split_ts.tzinfo:
        val_split_ts = val_split_ts.tz_convert(None)
    arrivals = compute_arrival_series(df_full, case_col="caseid", time_col="end_timestamp")
    arrivals = arrivals[arrivals.index < val_split_ts]
    arrival_model = ProphetArrivalModel().fit(arrivals)
    predicted_arrivals = arrival_model.predict(pd.DatetimeIndex(cc["test"].index).tz_localize(None))
    sos_df = build_sos_cases(
        predicted_arrivals,
        most_frequent_first_activity(known_df), most_frequent_first_resource(known_df),
        empirical_arrival_hour_sampler(known_df),
    )

    trainer_a = AmiriTrainer(train_df, val_df, test_df, run_name=f"{dataset}_seed{seed}",
                             params=params, output_dir=model_dir, dataset_dir=dataset_dir)
    rem_time_df = predict_amiri_plain_field(trainer_a, sos_df, inflight_df)
    event_log = rem_time_to_event_log(rem_time_df)

    cc_p, tt_p, m = compute_cc_tt_metrics(event_log, cc["test"], tt["test"])
    metrics = pd.DataFrame([
        dict(dataset=dataset, series="concurrent_cases", model="amiri", seed=seed, mae=m["cc_mae"], mse=m["cc_mse"]),
        dict(dataset=dataset, series="throughput_time", model="amiri", seed=seed, mae=m["tt_mae"], mse=m["tt_mse"]),
    ])
    pf_dir.mkdir(parents=True, exist_ok=True)
    metrics.to_csv(metrics_path, index=False)
    rem_time_df.to_csv(pf_dir / "rem_time.csv", index=False)
    print(f"  [done] {dataset}/seed_{seed}/pf: cc_mae={m['cc_mae']:.2f} tt_mae={m['tt_mae']:.2f}")


for name in SYNTH_DATASETS:
    print(f"\n{'='*60}\n{name} (amiri, synthetic, half)\n{'='*60}")
    for seed in NEW_SEEDS:
        run_amiri_half(name, is_real=False, seed=seed)
for name in REAL_DATASETS:
    print(f"\n{'='*60}\n{name} (amiri, ssd, half)\n{'='*60}")
    for seed in NEW_SEEDS:
        run_amiri_half(name, is_real=True, seed=seed)

for name in SYNTH_DATASETS:
    print(f"\n{'='*60}\n{name} (amiri, synthetic, pf)\n{'='*60}")
    for seed in NEW_SEEDS:
        run_amiri_pf(name, is_real=False, seed=seed)
for name in REAL_DATASETS:
    print(f"\n{'='*60}\n{name} (amiri, ssd, pf)\n{'='*60}")
    for seed in NEW_SEEDS:
        run_amiri_pf(name, is_real=True, seed=seed)
